In [33]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

from climate_year_selection_tool import add_custom_year, save_result, select_years


# Setup: define I/O filepaths, seasons

In [34]:
# Path of source data file (this one is generated by make_synthetic_data.py)
DATA_FILE = Path.cwd().parent / "data" / "synthetic_climate_data.csv"

# Define seasons for seasonal scoring
SEASONS = {"Winter": [10, 11, 12, 1, 2, 3], "Summer": [4, 5, 6, 7, 8, 9]}

# Results folder: results/all_example_usages/<timestamp>/
RESULTS_DIR = (
    Path.cwd().parent
    / "results"
    / "example_usage"
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# load reference dataset
df_ref = pd.read_csv(DATA_FILE, parse_dates=["Date"])

# Use-case 1: Selecting 5 representative years (default calendar years), for a single model

### Set up experiment

In [48]:
# Add year column
df_ref["year"] = df_ref["Date"].dt.year

# define experiment setup
experiment1_setup = {
    "variables": [
        "temp",
        "wind",
        "solar",
    ],  # variables to include in the scoring, i.e. on which variables the selected years should be representative. Must be columns in df.
    "n_select": 5,  # number of years to select
    "year_col": "year",  # column in df that contains the year information
    "scoring": "seasonal_wasserstein",  # default cost function
    "seasons": SEASONS,  # definition of seasons for seasonal scoring (only needed for seasonal scoring)
    "n_experiments": 5,  # number of times to repeat the selection with different random seeds (to get a distribution of scores)
    "cooling_rate": 0.98,  # cooling rate for simulated annealing (0-1). Closer to 1 means slower cooling, which can lead to better results but longer runtime.
    "max_iter": 2_000,  # maximum number of iterations for the simulated annealing algorithm. Higher means longer runtime, but potentially better results.
    "random_state": 42,  # random seed for reproducibility
}

### Run experiment

In [49]:
result = select_years(df_ref, **experiment1_setup)

Running 5 SA experiments in parallel...


SA experiments: 100%|██████████| 5/5 [00:09<00:00,  1.80s/it]


### Inspect results

In [50]:
print(f"Selected years        : {result.selected}")
print(f"Best score            : {result.score:.3f}")
print(f"# of iterations (best): {len(result.log_df)}")
print(f"Num experiments       : {len(result.all_runs)}")
print(f"All scores            : {[round(r['score'], 3) for r in result.all_runs]}")

Selected years        : [1906, 1909, 1928, 1947, 1979]
Best score            : 0.698
# of iterations (best): 436
Num experiments       : 5
All scores            : [0.698, 0.794, 0.803, 0.809, 0.81]


### Save results tot results directory

In [51]:
save_result(result, "exp1", experiment1_setup, RESULTS_DIR)

  → saved to /Users/bramvanduinen/Documents/climate_year_selection_tool_repo/results/example_usage/20260313_125426/exp1__n5__seasonal_wasserstein__nexp5__cr0.98__seed42_*


# Use-case 2: Custom year definition
For some applications (e.g. hydrology), calendar years might not be the most appropriate to select due to their cut-off in the middle of winter. For these applications, it's possible to perform selection on shifted, custom years. Here, we give an example with hydrological years from April 1st to March 31st. However, any shift can be applied.

In [52]:
# e.g. hydrological year 2025 runs from 2025-04-01 to 2026-03-31
df_ref = add_custom_year(
    df_ref, date_col="Date", start_month=4, year_col_name="hydro_year"
)

In [53]:
# Only change the year_col in the setup, to select based on the hydro_year instead of calendar year
experiment2_setup = experiment1_setup.copy()
experiment2_setup["year_col"] = "hydro_year"

In [55]:
# run experiment, inspect results and save
result_exp2 = select_years(df_ref, **experiment2_setup)

print(f"Selected years        : {result_exp2.selected}")
print(f"Best score            : {result_exp2.score:.3f}")
print(f"# of iterations (best): {len(result_exp2.log_df)}")
print(f"Num experiments       : {len(result_exp2.all_runs)}")
print(f"All scores            : {[round(r['score'], 3) for r in result_exp2.all_runs]}")

save_result(result_exp2, "exp2", experiment2_setup, RESULTS_DIR)

Running 5 SA experiments in parallel...


SA experiments: 100%|██████████| 5/5 [00:09<00:00,  1.96s/it]

Selected years        : [1937, 1943, 1974, 1975, 1976]
Best score            : 0.761
# of iterations (best): 436
Num experiments       : 5
All scores            : [0.761, 0.846, 0.858, 0.871, 0.979]
  → saved to /Users/bramvanduinen/Documents/climate_year_selection_tool_repo/results/example_usage/20260313_125426/exp2__n5__seasonal_wasserstein__nexp5__cr0.98__seed42__ychydro_year_*


# Use-case 3: Multiple models
Some climate datasets have multiple models, or an ensemble of different model initialisations. In those cases, just a selected year is ambigious. The selector can also include which model (or ensemble member), the chosen year is from. Here we demonstrate this simply by copying the dataset, with one called model_a and one called model_b.

In [56]:
# build multi-model dataset by concatenating the reference dataset with itself and adding a model column to distinguish the two
df_a = df_ref.assign(model="model_a")
df_b = df_ref.assign(model="model_b")
df_ref_multi = pd.concat([df_a, df_b], ignore_index=True)

In [ ]:
# Use the same setup as experiment 1, but specify the model column for multi-model datasets
experiment3_setup = experiment1_setup.copy()
experiment3_setup["model_col"] = (
    "model"  # column in df that contains the model information (for multi-model datasets)
)

In [60]:
# run experiment, inspect results and save
result_exp3 = select_years(df_ref_multi, **experiment3_setup)

print(f"Selected years        : {result_exp3.selected}")
print(f"Best score            : {result_exp3.score:.3f}")
print(f"# of iterations (best): {len(result_exp3.log_df)}")
print(f"Num experiments       : {len(result_exp3.all_runs)}")
print(f"All scores            : {[round(r['score'], 3) for r in result_exp3.all_runs]}")

save_result(result_exp3, "exp3", experiment3_setup, RESULTS_DIR)

Running 5 SA experiments in parallel...


SA experiments: 100%|██████████| 5/5 [00:18<00:00,  3.64s/it]

Selected years        : [('model_a', 1930), ('model_a', 1956), ('model_b', 1933), ('model_b', 1938), ('model_b', 1967)]
Best score            : 0.815
# of iterations (best): 436
Num experiments       : 5
All scores            : [0.815, 0.824, 0.833, 0.833, 0.894]
  → saved to /Users/bramvanduinen/Documents/climate_year_selection_tool_repo/results/example_usage/20260313_125426/exp3__n5__seasonal_wasserstein__nexp5__cr0.98__seed42_*
